# Train BPE Tokenizer on Legal Corpus

Trains the A1 BPE tokenizer on all 4 legal sources. Run once, save the JSON, reuse everywhere.

**Requirements:** Internet ON, HuggingFace secret `HF` set. GPU not needed.

## 1. HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

## 2. BPE Tokenizer Class

Your A1 implementation, optimized with word frequencies + incremental pair updates + checkpointing.

In [ ]:
import json
import os
import time
import random
from collections import defaultdict


class BPETokenizer:

    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos.extend(range(256))
        self.eow_id = len(self.itos)  # 260
        self.itos.append("</w>")
        self.merges = []
        self.vocab_size = len(self.itos)
        self._merge_rank = {}
        self._encode_cache = {}
        self._token_bytes = None
        self._ends_word = None

    # --- Training ---

    def fit(self, corpus, checkpoint_dir=None, checkpoint_every=1000):
        ckpt_path = None
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f"bpe_v{self.target_vocab_size}.ckpt.json")

        resumed_merge = 0
        word_data = None

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Found checkpoint at {ckpt_path}, resuming...")
            with open(ckpt_path, "r") as f:
                ckpt = json.load(f)
            self.merges = [((a, b), c) for (a, b), c in ckpt["merges"]]
            self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
            self.itos.extend(range(256))
            self.itos.append("</w>")
            for pair, new_id in self.merges:
                self.itos.append(pair)
            self.vocab_size = len(self.itos)
            resumed_merge = len(self.merges)
            word_data = []
            for tokens, freq in ckpt["words"]:
                word_data.append([tokens, freq])
            print(f"  Resumed after {resumed_merge} merges (vocab: {self.vocab_size})")

        if word_data is None:
            print("Counting word frequencies...")
            word_freq = {}
            for word in corpus.split():
                word_freq[word] = word_freq.get(word, 0) + 1
            print(f"  {len(word_freq):,} unique words")

            word_data = []
            for word, freq in word_freq.items():
                tokens = [b + 4 for b in word.encode("utf-8")]
                tokens.append(self.eow_id)
                word_data.append([tokens, freq])

        print("Building pair index...")
        pair_counts = defaultdict(int)
        pair_words = defaultdict(set)

        for idx, (tokens, freq) in enumerate(word_data):
            seen = defaultdict(int)
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                seen[pair] += 1
            for pair, cnt in seen.items():
                pair_counts[pair] += cnt * freq
                pair_words[pair].add(idx)

        num_merges_total = self.target_vocab_size - 261
        num_merges_left = num_merges_total - resumed_merge
        print(f"Learning {num_merges_left} merges (of {num_merges_total} total)...")
        t0 = time.time()

        for merge_i in range(num_merges_left):
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] < 2:
                break

            new_id = len(self.itos)
            bp_a, bp_b = best_pair
            affected = list(pair_words.get(best_pair, set()))

            for idx in affected:
                tokens, freq = word_data[idx]

                old_pairs = defaultdict(int)
                for i in range(len(tokens) - 1):
                    old_pairs[(tokens[i], tokens[i + 1])] += 1
                for p, cnt in old_pairs.items():
                    pair_counts[p] -= cnt * freq
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_words[p].discard(idx)
                    if not pair_words[p]:
                        pair_words.pop(p, None)

                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == bp_a and tokens[i + 1] == bp_b:
                        new_tokens.append(new_id)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                word_data[idx][0] = new_tokens

                new_pairs = defaultdict(int)
                for i in range(len(new_tokens) - 1):
                    new_pairs[(new_tokens[i], new_tokens[i + 1])] += 1
                for p, cnt in new_pairs.items():
                    pair_counts[p] = pair_counts.get(p, 0) + cnt * freq
                    pair_words[p].add(idx)

            self.merges.append((best_pair, new_id))
            self.itos.append(best_pair)
            self.vocab_size = len(self.itos)

            done = resumed_merge + merge_i + 1
            if done % 1000 == 0:
                elapsed = time.time() - t0
                rate = (merge_i + 1) / elapsed
                eta = (num_merges_left - merge_i - 1) / rate if rate > 0 else 0
                print(f"  {done}/{num_merges_total} merges | "
                      f"{elapsed:.0f}s elapsed | "
                      f"~{eta / 60:.1f} min remaining")

            if ckpt_path and (merge_i + 1) % checkpoint_every == 0:
                self._save_checkpoint(ckpt_path, word_data)

        if ckpt_path:
            self._save_checkpoint(ckpt_path, word_data)

        self._rebuild_caches()
        total_time = time.time() - t0
        print(f"Done. Final vocab: {self.vocab_size} | Time: {total_time / 60:.1f} min")
        return self

    def _save_checkpoint(self, path, word_data):
        ckpt = {
            "merges": [[[a, b], c] for (a, b), c in self.merges],
            "words": [[tokens, freq] for tokens, freq in word_data],
        }
        with open(path, "w") as f:
            json.dump(ckpt, f)

    # --- Encoding ---

    def _rebuild_caches(self):
        self._merge_rank = {}
        for rank, ((a, b), new_id) in enumerate(self.merges):
            self._merge_rank[(a, b)] = (rank, new_id)
        self._encode_cache = {}
        self._token_bytes = {x + 4: bytes([x]) for x in range(256)}
        self._token_bytes[self.eow_id] = b""
        self._ends_word = {i + 4: False for i in range(256)}
        self._ends_word[self.eow_id] = True
        for (a, b), c in self.merges:
            self._token_bytes[c] = self._token_bytes[a] + self._token_bytes[b]
            self._ends_word[c] = self._ends_word[b]

    def _encode_word(self, word):
        tokens = [b + 4 for b in word.encode("utf-8")]
        tokens.append(self.eow_id)
        while len(tokens) >= 2:
            best_rank = float("inf")
            best_idx = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                r = self._merge_rank.get(pair)
                if r is not None and r[0] < best_rank:
                    best_rank = r[0]
                    best_idx = i
            if best_idx == -1:
                break
            _, new_id = self._merge_rank[(tokens[best_idx], tokens[best_idx + 1])]
            tokens = tokens[:best_idx] + [new_id] + tokens[best_idx + 2:]
        return tokens

    def encode(self, text):
        if not self._merge_rank and self.merges:
            self._rebuild_caches()
        result = []
        for word in text.split():
            cached = self._encode_cache.get(word)
            if cached is not None:
                result.extend(cached)
            else:
                tokens = self._encode_word(word)
                self._encode_cache[word] = tokens
                result.extend(tokens)
        return result

    # --- Decoding ---

    def decode(self, tokens):
        if self._token_bytes is None:
            self._rebuild_caches()
        parts = []
        for i, token in enumerate(tokens):
            b = self._token_bytes.get(token)
            if b is not None:
                parts.append(b)
                if self._ends_word.get(token, False) and i < len(tokens) - 1:
                    parts.append(b" ")
        return b"".join(parts).decode("utf-8", errors="replace")

    # --- Save / Load ---

    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "target_vocab_size": self.target_vocab_size,
            "merges": [[[a, b], c] for (a, b), c in self.merges],
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Tokenizer saved to {path} ({self.vocab_size} tokens)")

    @classmethod
    def load(cls, path):
        with open(path, "r") as f:
            data = json.load(f)
        tok = cls(data["target_vocab_size"])
        tok.merges = [((a, b), c) for (a, b), c in data["merges"]]
        for pair, new_id in tok.merges:
            tok.itos.append(pair)
        tok.vocab_size = len(tok.itos)
        tok._rebuild_caches()
        return tok

    @property
    def pad_token_id(self):
        return 0

    @property
    def unk_token_id(self):
        return 1

    @property
    def bos_token_id(self):
        return 2

    @property
    def eos_token_id(self):
        return 3

    def __len__(self):
        return self.vocab_size


print("BPETokenizer class loaded.")

## 3. Configuration

In [ ]:
VOCAB_SIZE = 16384
CHECKPOINT_EVERY = 1000
SEED = 42
random.seed(SEED)

TARGET_WORDS_PER_SOURCE = 15_000_000

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

OUTPUT_DIR = "/kaggle/working"
CKPT_DIR = os.path.join(OUTPUT_DIR, "bpe_checkpoints")
TOKENIZER_PATH = os.path.join(OUTPUT_DIR, f"legal_bpe_{VOCAB_SIZE}.json")

print(f"Vocab size: {VOCAB_SIZE}")
print(f"Target words per source: {TARGET_WORDS_PER_SOURCE:,}")
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Output: {TOKENIZER_PATH}")

## 4. Download and Sample from All 4 Sources

In [ ]:
from huggingface_hub import hf_hub_download

try:
    import ijson
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "ijson"])
    import ijson

all_texts = []

for train_file in TRAIN_FILES:
    source_name = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
    print(f"\n[{source_name}] Downloading...")

    file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=train_file,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    print(f"[{source_name}] Streaming records...")
    source_texts = []
    word_count = 0

    with open(file_path, "rb") as f:
        for record in ijson.items(f, "item"):
            text = record["text"]
            words = len(text.split())
            source_texts.append(text)
            word_count += words
            if word_count >= TARGET_WORDS_PER_SOURCE:
                break

    all_texts.extend(source_texts)
    print(f"[{source_name}] Sampled {len(source_texts):,} records ({word_count:,} words)")

random.shuffle(all_texts)
corpus = " ".join(all_texts)
total_words = len(corpus.split())
print(f"\nCombined corpus: {total_words:,} words, {len(corpus) / 1e6:.1f} MB")

## 5. Train Tokenizer

This takes ~15-25 minutes. Checkpoints every 1000 merges.

**If Kaggle times out:** Just re-run this cell. It will detect the checkpoint and resume.

In [ ]:
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)

t0 = time.time()
tokenizer.fit(corpus, checkpoint_dir=CKPT_DIR, checkpoint_every=CHECKPOINT_EVERY)
total_time = time.time() - t0

print(f"\nTotal training time: {total_time / 60:.1f} min")

## 6. Save Final Tokenizer

In [ ]:
tokenizer.save(TOKENIZER_PATH)
print(f"\nSaved to: {TOKENIZER_PATH}")
print(f"File size: {os.path.getsize(TOKENIZER_PATH) / 1e6:.1f} MB")

## 7. Sanity Checks

In [ ]:
# Reload from file to verify save/load works
tok = BPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded vocab size: {len(tok)}")

# Test roundtrip on diverse legal text
test_texts = [
    "The Supreme Court held that the petition filed by the appellant was dismissed.",
    "Section 302 of the Indian Penal Code prescribes punishment for murder.",
    "The High Court of Judicature at Allahabad ruled in favor of the respondent.",
    "The tribunal directed the employer to reinstate the workman with full back wages.",
    "Under Article 226 of the Constitution of India the writ petition is maintainable.",
]

print("\n--- Roundtrip Tests ---")
for text in test_texts:
    encoded = tok.encode(text)
    decoded = tok.decode(encoded)
    fertility = len(encoded) / len(text.split())
    match = decoded.strip() == text
    status = "PASS" if match else "FAIL"
    print(f"\n  [{status}] {text[:60]}...")
    print(f"         {len(encoded)} tokens, {fertility:.2f} tokens/word")
    if not match:
        print(f"         Got: {decoded.strip()[:80]}")

In [ ]:
# Corpus-level fertility stats
sample_text = " ".join(all_texts[:50])
sample_words = sample_text.split()
bpe_tokens = tok.encode(sample_text)
byte_count = len(sample_text.encode("utf-8"))

print("--- Fertility Stats (50-record sample) ---")
print(f"  Words:           {len(sample_words):,}")
print(f"  BPE tokens:      {len(bpe_tokens):,}")
print(f"  Tokens/word:     {len(bpe_tokens)/len(sample_words):.2f}")
print(f"  Raw bytes:       {byte_count:,}")
print(f"  Compression:     {byte_count/len(bpe_tokens):.2f}x vs raw bytes")
print(f"\nTokenizer ready at: {TOKENIZER_PATH}")
print("Save this file as a Kaggle dataset to reuse in all experiment notebooks.")

## 8. How to Use in Other Notebooks

After saving `legal_bpe_16384.json` as a Kaggle dataset, use it like this:

```python
# Paste the BPETokenizer class cell, then:
tokenizer = BPETokenizer.load("/kaggle/input/your-dataset-name/legal_bpe_16384.json")

# Encode text to token IDs
token_ids = tokenizer.encode("The Supreme Court held that...")

# Decode token IDs back to text
text = tokenizer.decode(token_ids)

# Vocab size (for model config)
vocab_size = len(tokenizer)  # 16384

# Special token IDs
tokenizer.pad_token_id  # 0
tokenizer.bos_token_id  # 2
tokenizer.eos_token_id  # 3
```